In [1]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import cv2

In [2]:
model_path = 'face_landmarker.task'

In [3]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create a face landmarker instance with the video mode:
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO)

In [4]:
def video_to_numpy(video_path, target_size=(224, 224), convert_rgb=True):
    """
    Reads a video clip from the DAiSEE dataset and returns a 4D NumPy array.
    Shape: (Num_Frames, Height, Width, Channels)
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")
        
    frames = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break  # Break the loop if the video ends or cannot be read
            
        # Optional: Resize the frame to reduce memory usage
        if target_size:
            frame = cv2.resize(frame, target_size)
            
        # OpenCV reads frames in BGR format by default; convert to RGB
        if convert_rgb:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
        frames.append(frame)
        
    cap.release()
    
    # Stack individual frames into a single 4D NumPy array
    video_array = np.stack(frames, axis=0)
    return video_array

# Example usage:
video_file = "DAiSEE/DataSet/Train/110001/1100011002/1100011002.avi"
video_np = video_to_numpy(video_file, target_size=(224, 224))

print("NumPy Array Shape:", video_np.shape)  # Output example: (300, 224, 224, 3)
print("Data Type:", video_np.dtype)  

AttributeError: module 'cv2' has no attribute 'VideoCapture'

In [5]:
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=video_np[0])

NameError: name 'video_np' is not defined

In [6]:
with FaceLandmarker.create_from_options(options) as landmarker:
    face_landmarker_result = landmarker.detect_for_video(mp_image, 1)

: 

## Train / validation / test face-landmark datasets

The cells below are self-contained: they build a manifest for each of
DAiSEE's three official splits (`Train`, `Validation`, `Test`), run every
clip through the face landmarker to get a fixed-length landmark sequence,
cache each clip's sequence as a `.npy` file, and wrap the three caches in
PyTorch `Dataset`/`DataLoader` objects. They reuse `FaceLandmarker`,
`options`, `mp`, `cv2`, and `np` from the cells above, and don't depend on
the scratch cells that crashed the kernel earlier.

In [ ]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader


In [ ]:
# --- Paths & config -----------------------------------------------------
DAISEE_ROOT = Path("DAiSEE")
LABELS_DIR = DAISEE_ROOT / "Labels"
DATASET_DIR = DAISEE_ROOT / "DataSet"

# split key -> (DAiSEE's folder name, labels CSV)
SPLITS = {
    "train": ("Train", LABELS_DIR / "TrainLabels.csv"),
    "val": ("Validation", LABELS_DIR / "ValidationLabels.csv"),
    "test": ("Test", LABELS_DIR / "TestLabels.csv"),
}

LANDMARKS_DIR = Path("landmarks")  # ml/landmarks/{train,val,test}/<clip_id>.npy
LANDMARKS_DIR.mkdir(exist_ok=True)

LABEL_COLUMNS = ["Boredom", "Engagement", "Confusion", "Frustration"]

NUM_FRAMES = 30  # frames sampled per clip (uniform stride) -> fixed-length sequence


In [ ]:
def build_video_index(dataset_split_dir: Path) -> dict[str, Path]:
    """Map ClipID (e.g. '1100011002.avi') -> full path to the .avi file.

    Walking the tree once and indexing by filename is more robust than
    reconstructing the path from the subject-id prefix, since a few subject
    folders in DAiSEE don't follow the 6-digit convention.
    """
    index = {}
    for avi_path in dataset_split_dir.rglob("*.avi"):
        index[avi_path.name] = avi_path
    return index


def load_split_manifest(split_key: str) -> pd.DataFrame:
    """Load a split's label CSV and attach the resolved video path to each row."""
    subdir_name, labels_csv = SPLITS[split_key]
    df = pd.read_csv(labels_csv)
    df.columns = df.columns.str.strip()  # "Frustration " has a trailing space in the source CSV

    video_index = build_video_index(DATASET_DIR / subdir_name)
    df["video_path"] = df["ClipID"].map(video_index)

    missing = df["video_path"].isna().sum()
    if missing:
        print(f"[{split_key}] warning: {missing} clip(s) listed in the labels CSV were not found on disk")
        df = df.dropna(subset=["video_path"]).reset_index(drop=True)

    return df


train_manifest = load_split_manifest("train")
val_manifest = load_split_manifest("val")
test_manifest = load_split_manifest("test")

for _name, _df in [("train", train_manifest), ("val", val_manifest), ("test", test_manifest)]:
    print(f"{_name}: {len(_df)} clips")


In [ ]:
def extract_landmarks(video_path: Path, num_frames: int = NUM_FRAMES) -> "np.ndarray | None":
    """Sample `num_frames` evenly-spaced frames from a clip and run the face
    landmarker on each. Returns an array of shape (num_frames, num_landmarks, 3)
    of (x, y, z) landmark coordinates, or None if no face was ever detected
    in the sampled frames.

    A fresh FaceLandmarker is created per video: VIDEO running mode requires
    strictly increasing timestamps on a single landmarker instance, and giving
    each clip its own timeline (starting at t=0) keeps that simple.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise IOError(f"Cannot open video file: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return None

    sample_indices = np.linspace(0, frame_count - 1, num=num_frames).round().astype(int)
    sample_set = set(sample_indices.tolist())

    frames_by_index = {}
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx in sample_set:
            frames_by_index[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        idx += 1
    cap.release()

    landmarks_by_index = {}
    with FaceLandmarker.create_from_options(options) as landmarker:
        for i, frame_idx in enumerate(sample_indices):
            rgb = frames_by_index.get(int(frame_idx))
            if rgb is None:
                continue
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(i * (1000 / fps))
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            if result.face_landmarks:
                face = result.face_landmarks[0]
                landmarks_by_index[i] = np.array([[lm.x, lm.y, lm.z] for lm in face], dtype=np.float32)

    if not landmarks_by_index:
        return None

    n_points = len(next(iter(landmarks_by_index.values())))
    sequence = np.full((num_frames, n_points, 3), np.nan, dtype=np.float32)
    for i, lm in landmarks_by_index.items():
        sequence[i] = lm

    # Fill frames with no detection from the nearest earlier detected frame so
    # the cached sequence has no NaNs (a blink, head turn, or a moment fully
    # out of frame happens occasionally in these webcam clips).
    detected = ~np.isnan(sequence[:, 0, 0])
    if detected.any():
        last_good = np.where(detected)[0][0]
        for i in range(num_frames):
            if detected[i]:
                last_good = i
            else:
                sequence[i] = sequence[last_good]

    return sequence


In [ ]:
def process_split(split_key: str, manifest: pd.DataFrame, limit: "int | None" = None) -> pd.DataFrame:
    """Extract + cache landmarks for every clip in a split.

    Resumable: a clip whose .npy already exists on disk is skipped, so a
    partially-run split can be re-run safely (e.g. after raising `limit`,
    or after fixing an error partway through the full split).
    """
    out_dir = LANDMARKS_DIR / split_key
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = manifest.iloc[:limit] if limit else manifest
    records = []
    failed = []

    for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=split_key):
        clip_id = Path(row.ClipID).stem
        npy_path = out_dir / f"{clip_id}.npy"

        if not npy_path.exists():
            try:
                sequence = extract_landmarks(Path(row.video_path))
            except Exception as exc:
                failed.append((row.ClipID, str(exc)))
                continue
            if sequence is None:
                failed.append((row.ClipID, "no face detected in any sampled frame"))
                continue
            np.save(npy_path, sequence)

        records.append({
            "clip_id": clip_id,
            "npy_path": str(npy_path),
            "Boredom": row.Boredom,
            "Engagement": row.Engagement,
            "Confusion": row.Confusion,
            "Frustration": row.Frustration,
        })

    if failed:
        print(f"[{split_key}] {len(failed)} clip(s) skipped (no face detected / read error)")

    split_df = pd.DataFrame.from_records(records)
    split_df.to_csv(LANDMARKS_DIR / f"{split_key}_manifest.csv", index=False)
    return split_df


Run the cell below to build the caches. `LIMIT` caps how many clips per
split get processed — start with a small number to sanity-check the
pipeline before committing to a full run: the full splits are ~5.4k / 1.4k
/ 1.8k clips for train/val/test, and every sampled frame goes through the
face landmarker, so processing everything will take a while. Re-running
with a larger (or `None`) `LIMIT` picks up where it left off, since already
-cached clips are skipped.

In [ ]:
LIMIT = 20  # set to None to process every clip in each split

train_landmarks = process_split("train", train_manifest, limit=LIMIT)
val_landmarks = process_split("val", val_manifest, limit=LIMIT)
test_landmarks = process_split("test", test_manifest, limit=LIMIT)

len(train_landmarks), len(val_landmarks), len(test_landmarks)


In [ ]:
class DaiseeLandmarksDataset(Dataset):
    """Loads cached face-landmark sequences for one DAiSEE split.

    Each item is (landmarks, labels):
      - landmarks: FloatTensor, shape (NUM_FRAMES, num_landmarks, 3) by
        default -- keeping (x, y, z) grouped per landmark, which matters if
        the model treats landmarks as points/a graph or normalizes z
        (relative depth) separately from x/y. Pass flatten=True to instead
        get (NUM_FRAMES, num_landmarks * 3), e.g. for a plain LSTM/Transformer
        that just wants a flat per-frame feature vector.
      - labels: LongTensor of the 4 DAiSEE affect scores (each 0-3), in the
        order Boredom, Engagement, Confusion, Frustration -- unless `target`
        names a single column (e.g. "Engagement") to return instead.
    """

    def __init__(self, manifest: pd.DataFrame, target: "str | None" = None, flatten: bool = False):
        self.manifest = manifest.reset_index(drop=True)
        self.target = target
        self.flatten = flatten

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, idx: int):
        row = self.manifest.iloc[idx]
        sequence = np.load(row["npy_path"])  # (NUM_FRAMES, num_landmarks, 3)
        if self.flatten:
            sequence = sequence.reshape(sequence.shape[0], -1)
        landmarks = torch.from_numpy(sequence).float()

        if self.target:
            labels = torch.tensor(row[self.target], dtype=torch.long)
        else:
            labels = torch.tensor(row[LABEL_COLUMNS].to_numpy(dtype="int64"), dtype=torch.long)

        return landmarks, labels


In [ ]:
train_dataset = DaiseeLandmarksDataset(train_landmarks)
val_dataset = DaiseeLandmarksDataset(val_landmarks)
test_dataset = DaiseeLandmarksDataset(test_landmarks)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

landmarks_batch, labels_batch = next(iter(train_loader))
print("landmarks batch:", landmarks_batch.shape)  # (B, NUM_FRAMES, num_landmarks, 3)
print("labels batch:", labels_batch.shape)          # (B, 4)
